# NCCT Fixed Dataset Split

Notebook **terpisah** dari `NCCT_MultiStage_Classification_V1.ipynb` — khusus membuat **fixed dataset split**
agar hasil training **reproducible** (tidak bergantung pada split on-the-fly setiap kali run).

> Di `V1`, split dilakukan *on the fly* (`patient_level_split()` dipanggil ulang tiap run), sehingga komposisi
> train/val/test bisa berubah antar run. Notebook ini menyelesaikan masalah itu: split dibuat **sekali, FIXED**,
> lalu hasilnya di-*zip* dan di-upload manual ke Google Drive untuk dipakai di notebook training.

## Output notebook ini

Folder `NCCT_fixed_split/` yang berisi:

| Isi | Keterangan |
|-----|------------|
| `metadata/stage1_lvo/{train,val,test}.csv` | Stage 1 — semua pasien (LVO vs Non-LVO) |
| `metadata/stage2_laterality/{train,val,test}.csv` | Stage 2 — pasien LVO (Left vs Right) |
| `metadata/stage3_etiology/{train,val,test}.csv` | Stage 3 — pasien LVO (Cardio vs Non-Cardio) |
| `train/`, `val/`, `test/` | Citra yang sudah dibagi per set (struktur folder asli dipertahankan) |
| `split_summary.json` | Ringkasan split (jumlah pasien & citra per set) |

Kemudian seluruh folder di-*zip* menjadi **`NCCT_fixed_split.zip`** → upload manual ke Google Drive.

> **Catatan:** `NCCT_MultiStage_Classification_V1.ipynb` **tidak diubah**. Sel bernomor 1-20 di bawah ini berjalan
> mandiri, dimulai dari download dataset asli sampai menghasilkan zip.

## Konsep Split

Split dilakukan **per pasien** (bukan per citra) untuk mencegah *data leakage* — slice-slice satu pasien sangat
mirip, sehingga tidak boleh tersebar ke train & test sekaligus.

**Poin kunci — split GLOBAL:**

1. Semua pasien di-split **sekali** ke `train / val / test` (stratified, seed tetap).
2. Setiap stage tinggal **memotong** split global itu ke subset-nya:
   - **Stage 1** → semua pasien
   - **Stage 2** → pasien LVO yang punya label laterality
   - **Stage 3** → pasien LVO yang punya label etiologi
3. Konsekuensinya: pasien yang masuk test di Stage 1 **otomatis juga test** di Stage 2 & 3.
   Ini membuat evaluasi pipeline end-to-end (Stage 1 → 2 → 3) valid, dan split **konsisten antar stage**
   (tidak seperti V1 yang menyplit tiap stage secara independen).

### 1. Install & Import Library

In [ ]:
!pip install -q gdown openpyxl scikit-learn pandas numpy

### 2. Konfigurasi

Ubah `MODE` menjadi `'local'` jika menjalankan di komputer sendiri dengan dataset sample, atau `'colab'` untuk dataset penuh dari Google Drive.

In [ ]:
import os, re, glob, shutil, json
import numpy as np
import pandas as pd

# ============================================================
#  KONFIGURASI
# ============================================================
MODE = 'colab'      # 'colab' → download dataset penuh dari Google Drive
                    # 'local' → pakai dataset sample di folder lokal (testing)

GDRIVE_FILE_ID = '16WChY76ppX0WPBlfud6tLWrSCi9psdxt'  # ID zip dataset di Drive
DATA_ROOT      = 'dataset'                            # folder hasil ekstrak (mode colab)
SAMPLE_DIR     = 'NVO-classification-dataset-sample'  # dataset sample (mode local)

LVO_SUBDIR = 'selected_ncct'   # 'selected_ncct' → slice NCCT dgn stroke terlihat (default)
                               # 'selected_lvo'  → slice dgn overlay mask
                               # 'normal'        → slice sehat

# ===== Konfigurasi SPLIT (FIXED) =====
SEED        = 42                # seed TETAP → hasil split selalu identik tiap run
VAL_FRAC    = 0.15              # proporsi pasien utk validasi
TEST_FRAC   = 0.15              # proporsi pasien utk test
SPLIT_DIR   = 'NCCT_fixed_split'      # folder output (isi zip)
SPLIT_ZIP   = 'NCCT_fixed_split.zip'  # file zip → upload ke Google Drive
COPY_IMAGES = True              # True → salin citra ke train/val/test (zip self-contained)


### 3. Download & Extract Dataset

(Colab) Unduh zip dataset dari Google Drive lalu ekstrak ke `DATA_ROOT`. Di mode `local`, langsung pakai `SAMPLE_DIR`.

In [ ]:
if MODE == 'colab':
    !gdown {GDRIVE_FILE_ID}
    zips = glob.glob('*.zip')
    print('Zip hasil download:', zips)
    if not zips:
        raise RuntimeError('Tidak ada file zip. Periksa GDRIVE_FILE_ID.')
    zip_src = zips[0]                      # nama bisa Classification_LVO.zip dsb.
    !unzip -q -o {zip_src} -d {DATA_ROOT}
else:
    DATA_ROOT = SAMPLE_DIR
    print('Mode local -> pakai dataset sample:', DATA_ROOT)

print('Isi DATA_ROOT:', sorted(os.listdir(DATA_ROOT)))

### 4. Membangun Metadata (identik dengan V1)

Tabel metadata: **1 baris = 1 citra**, lengkap dengan label untuk 3 stage.

| Kolom | Stage | Sumber |
|-------|-------|--------|
| `label_lvo` | 1 | `Has LVO` (annotasi) / folder `NON-LVO` → Non-LVO |
| `label_laterality` | 2 | `Laterality (Left/Right)` |
| `label_etiology` | 3 | `Stroke Etiology` |

In [ ]:
def build_metadata(data_root, lvo_subdir='selected_ncct'):
    """Bangun tabel metadata: 1 baris = 1 citra, lengkap label untuk 3 stage."""
    lat_df  = pd.read_excel(os.path.join(data_root, 'LVO_Laterality_Annotations.xlsx'))
    etio_df = pd.read_excel(os.path.join(data_root, 'Stroke_Etiology_Annotations.xlsx'))
    annot = lat_df.merge(etio_df, on='Subject', how='left')

    def get_annot(subject):
        row = annot[annot['Subject'] == subject]
        if len(row) == 0:
            return None
        r = row.iloc[0]
        return {
            'is_lvo':     str(r['Has LVO']).strip().lower() == 'yes',
            'laterality': r.get('Laterality (Left/Right)'),
            'etiology':   r.get('Stroke Etiology'),
        }

    records, IMG_EXT = [], ('.jpg', '.jpeg', '.png', '.bmp')

    # ---- Pasien LVO : LVO/pXX_sub-strokeXXXX/{lvo_subdir}/*.jpg ----
    lvo_root = os.path.join(data_root, 'LVO')
    if os.path.isdir(lvo_root):
        for pdir in sorted(os.listdir(lvo_root)):
            ppath = os.path.join(lvo_root, pdir)
            if not os.path.isdir(ppath):
                continue
            m = re.search(r'(sub-stroke\d+)', pdir)
            subject = m.group(1) if m else pdir
            a = get_annot(subject)
            if a is None:      # di folder LVO tapi tak ada di annotasi -> anggap LVO
                a = {'is_lvo': True, 'laterality': None, 'etiology': None}
            img_dir = os.path.join(ppath, lvo_subdir)
            if not os.path.isdir(img_dir):
                print(f'  [skip] folder tidak ditemukan: {img_dir}')
                continue
            for f in sorted(os.listdir(img_dir)):
                if f.lower().endswith(IMG_EXT):
                    records.append(dict(image_path=os.path.join(img_dir, f),
                                        subject=subject, **a))

    # ---- Pasien Non-LVO : NON-LVO/lacunarCT-XXX/*.jpg ----
    nonlvo_root = os.path.join(data_root, 'NON-LVO')
    if os.path.isdir(nonlvo_root):
        for pdir in sorted(os.listdir(nonlvo_root)):
            ppath = os.path.join(nonlvo_root, pdir)
            if not os.path.isdir(ppath):
                continue
            for f in sorted(os.listdir(ppath)):
                if f.lower().endswith(IMG_EXT):
                    records.append(dict(image_path=os.path.join(ppath, f),
                                        subject=pdir, is_lvo=False,
                                        laterality=None, etiology=None))

    df = pd.DataFrame(records)
    df['label_lvo']        = df['is_lvo'].astype(int)                  # 1=LVO, 0=Non-LVO
    df['label_laterality'] = df['laterality'].map({'Left': 0, 'Right': 1})
    df['label_etiology']   = df['etiology'].map({'Non-Cardioembolic': 0, 'Cardioembolic': 1})
    return df

meta = build_metadata(DATA_ROOT, lvo_subdir=LVO_SUBDIR)
print(f'Total citra : {len(meta)}')
print(f'Total pasien: {meta["subject"].nunique()}')
display(meta.head())

### 5. Fixed Split — Global per Pasien

Fungsi inti: `global_patient_split()` mengembalikan 3 himpunan pasien (`train/val/test`) dengan seed tetap.

In [ ]:
from sklearn.model_selection import train_test_split


def build_patient_table(meta):
    """1 baris = 1 PASIEN + label tiap stage (nilai pertama yg tersedia)."""
    return (meta.groupby('subject', as_index=False)
                .agg(is_lvo=('label_lvo', 'first'),
                     laterality=('laterality', 'first'),
                     etiology=('etiology', 'first')))


def global_patient_split(meta, val_frac=VAL_FRAC, test_frac=TEST_FRAC, seed=SEED):
    """Split SEMUA pasien SEKALI (global), stratified & FIXED (seed tetap).

    Mengapa GLOBAL?
    - Pasien yg masuk test di Stage 1 otomatis juga test di Stage 2 & 3
      -> konsisten antar stage, evaluasi end-to-end valid, tanpa data leakage.
    - Satu pasien TIDAK PERNAH tersebar di >1 set.

    Stratifikasi memakai kombinasi (is_lvo, laterality, etiology) agar distribusi
    kelas tiap stage tetap terjaga proporsional di train/val/test.
    """
    p = build_patient_table(meta)
    strat = (p['is_lvo'].astype(str) + '_' +
             p['laterality'].fillna('None').astype(str) + '_' +
             p['etiology'].fillna('None').astype(str))
    idx = np.arange(len(p))

    try:
        tr_idx, te_idx = train_test_split(idx, test_size=test_frac,
                                          random_state=seed, stratify=strat)
        tr_idx2, va_idx = train_test_split(tr_idx,
                                           test_size=val_frac / (1 - test_frac),
                                           random_state=seed + 1,
                                           stratify=strat.iloc[tr_idx])
        train_pos, val_pos, test_pos = tr_idx[tr_idx2], tr_idx[va_idx], te_idx
    except ValueError:
        # Dataset terlalu kecil utk stratifikasi -> fallback shuffle biasa
        rng = np.random.RandomState(seed)
        idx2 = idx.copy(); rng.shuffle(idx2)
        n = len(idx2)
        n_te = max(1, int(round(n * test_frac)))
        n_va = max(1, int(round(n * val_frac)))
        while n_va + n_te >= n and n_va > 0 and n_te > 0:
            if n_te >= n_va:
                n_te -= 1
            else:
                n_va -= 1
        test_pos, val_pos, train_pos = idx2[:n_te], idx2[n_te:n_te + n_va], idx2[n_te + n_va:]

    train_subj = set(p.iloc[train_pos]['subject'])
    val_subj   = set(p.iloc[val_pos]['subject'])
    test_subj  = set(p.iloc[test_pos]['subject'])
    return train_subj, val_subj, test_subj


def apply_split(df, train_subj, val_subj, test_subj):
    """Bagi DataFrame per citra berdasarkan himpunan pasien per set."""
    return (df[df['subject'].isin(train_subj)].copy(),
            df[df['subject'].isin(val_subj)].copy(),
            df[df['subject'].isin(test_subj)].copy())

In [ ]:
# ===== Terapkan split global ke tiap stage =====
train_subj, val_subj, test_subj = global_patient_split(meta)
print(f'Jumlah PASIEN -> Train: {len(train_subj)} | Val: {len(val_subj)} | Test: {len(test_subj)}')

# ---- Stage 1 : semua pasien (LVO + Non-LVO) ----
s1 = meta[['image_path', 'subject', 'label_lvo']].rename(columns={'label_lvo': 'label'})

# ---- Stage 2 : hanya pasien LVO yg punya label laterality ----
s2 = (meta[meta['label_lvo'] == 1]
      .dropna(subset=['label_laterality'])
      [['image_path', 'subject', 'label_laterality']]
      .rename(columns={'label_laterality': 'label'}))
s2['label'] = s2['label'].astype(int)

# ---- Stage 3 : hanya pasien LVO yg punya label etiologi ----
s3 = (meta[meta['label_lvo'] == 1]
      .dropna(subset=['label_etiology'])
      [['image_path', 'subject', 'label_etiology']]
      .rename(columns={'label_etiology': 'label'}))
s3['label'] = s3['label'].astype(int)

splits = {
    'stage1_lvo':        apply_split(s1, train_subj, val_subj, test_subj),
    'stage2_laterality': apply_split(s2, train_subj, val_subj, test_subj),
    'stage3_etiology':   apply_split(s3, train_subj, val_subj, test_subj),
}

print('\nDistribusi citra & pasien per stage:')
for name, (tr, va, te) in splits.items():
    print(f'{name:18s} | citra  T:{len(tr):4d} V:{len(va):3d} Te:{len(te):3d} '
          f'| pasien T:{tr["subject"].nunique():3d} V:{va["subject"].nunique():3d} Te:{te["subject"].nunique():3d}')

### 6. Simpan CSV Metadata per Stage

`image_path` disimpan sebagai **path relatif** sehingga zip portabel: saat di-extract di lokasi mana pun, path tetap valid.

- `COPY_IMAGES=True` → path berbentuk `train/LVO/p1_sub-stroke0001/selected_ncct/03.jpg` (citra ada di dalam `SPLIT_DIR`).
- `COPY_IMAGES=False` → path berbentuk `LVO/p1_sub-stroke0001/selected_ncct/03.jpg` (relatif terhadap `DATA_ROOT`).

In [ ]:
# ===== Simpan CSV metadata per stage =====
os.makedirs(os.path.join(SPLIT_DIR, 'metadata'), exist_ok=True)

for name, (tr, va, te) in splits.items():
    d = os.path.join(SPLIT_DIR, 'metadata', name)
    os.makedirs(d, exist_ok=True)
    for split_name, df in [('train', tr), ('val', va), ('test', te)]:
        out = df.copy()
        if COPY_IMAGES:
            out['image_path'] = out['image_path'].apply(
                lambda p: os.path.join(split_name, os.path.relpath(p, DATA_ROOT)))
        else:
            out['image_path'] = out['image_path'].apply(
                lambda p: os.path.relpath(p, DATA_ROOT))
        out.to_csv(os.path.join(d, f'{split_name}.csv'), index=False)

print('CSV metadata tersimpan di:', os.path.join(SPLIT_DIR, 'metadata'))
for name in splits:
    for sp in ('train', 'val', 'test'):
        fp = os.path.join(SPLIT_DIR, 'metadata', name, f'{sp}.csv')
        print(f'  {os.path.relpath(fp)}  ({len(pd.read_csv(fp))} baris)')

### 7. Salin Citra ke `train/` / `val/` / `test/`

Struktur folder asli dipertahankan. Cukup menyalin citra **Stage 1** (yang mencakup semua citra) — citra stage 2 & 3 adalah subset-nya, jadi tidak perlu disalin ulang.

> Hardlink dipakai bila memungkinkan (hemat ruang & cepat); fallback ke `shutil.copy2` bila beda filesystem.

In [ ]:
# ===== Salin citra ke train/val/test =====
def link_or_copy(src, dst):
    """Hardlink kalau bisa (hemat ruang & cepat), fallback salin biasa."""
    os.makedirs(os.path.dirname(dst), exist_ok=True)
    try:
        os.link(src, dst)
    except OSError:
        shutil.copy2(src, dst)

if COPY_IMAGES:
    total = 0
    tr, va, te = splits['stage1_lvo']     # citra Stage 1 = SEMUA citra; stage 2/3 subset-nya
    for split_name, df in [('train', tr), ('val', va), ('test', te)]:
        for _, r in df.iterrows():
            rel = os.path.relpath(r['image_path'], DATA_ROOT)   # mis. LVO/p1_.../selected_ncct/03.jpg
            link_or_copy(r['image_path'], os.path.join(SPLIT_DIR, split_name, rel))
            total += 1
    print(f'Selesai. Total citra disalin ke {SPLIT_DIR}/{{train,val,test}}: {total}')
else:
    print('COPY_IMAGES=False -> lewati penyalinan citra (hanya CSV).')

### 8. Verifikasi Split

Empat pemeriksaan: (1) bebas leakage antar set, (2) stage 2 & 3 konsisten dengan split global, (3) reproducible (seed sama → hasil identik), (4) distribusi kelas per stage per set.

In [ ]:
# ===== Verifikasi =====
# 1) Tidak ada pasien yg sama di lebih dari satu set (bebas leakage)
assert train_subj.isdisjoint(val_subj) and train_subj.isdisjoint(test_subj) and val_subj.isdisjoint(test_subj), \
    'LEAKAGE: ada pasien yg sama di >1 set!'
print('✔ Tidak ada pasien yg sama di train/val/test (bebas data leakage).')

# 2) Split konsisten antar stage (stage 2 & 3 subset dari split global)
for name, (tr, va, te) in splits.items():
    if name == 'stage1_lvo':
        continue
    assert set(tr['subject']).issubset(train_subj), f'{name}: ada pasien train di luar split global!'
    assert set(te['subject']).issubset(test_subj),  f'{name}: ada pasien test di luar split global!'
print('✔ Stage 2 & 3 merupakan subset konsisten dari split global Stage 1.')

# 3) Reproducible: split dijalankan ulang -> hasil identik
t2, v2, te2 = global_patient_split(meta)
assert train_subj == t2 and val_subj == v2 and test_subj == te2, 'Split TIDAK reproducible!'
print(f'✔ Split reproducible (dijalankan 2x dgn seed={SEED} -> identik).')

# 4) Distribusi kelas per stage per set
for name, (tr, va, te) in splits.items():
    print(f'\n=== {name} ===')
    for split_name, df in [('train', tr), ('val', va), ('test', te)]:
        print(f'  {split_name:6s} (citra={len(df):4d}): {df["label"].value_counts().sort_index().to_dict()}')

# 5) Simpan ringkasan split
summary = {
    'seed': SEED, 'val_frac': VAL_FRAC, 'test_frac': TEST_FRAC,
    'lvo_subdir': LVO_SUBDIR, 'copy_images': COPY_IMAGES,
    'total_patients': {'train': len(train_subj), 'val': len(val_subj), 'test': len(test_subj)},
    'stages': {name: {'train_images': len(tr), 'val_images': len(va), 'test_images': len(te)}
               for name, (tr, va, te) in splits.items()},
}
with open(os.path.join(SPLIT_DIR, 'split_summary.json'), 'w') as f:
    json.dump(summary, f, indent=2)
print('\nRingkasan tersimpan di split_summary.json')

### 9. Buat ZIP untuk Upload ke Google Drive

In [ ]:
# ===== Buat ZIP utk di-upload ke Google Drive =====
if os.path.exists(SPLIT_ZIP):
    os.remove(SPLIT_ZIP)
shutil.make_archive(os.path.splitext(SPLIT_ZIP)[0], 'zip', root_dir=SPLIT_DIR)
size_mb = os.path.getsize(SPLIT_ZIP) / (1024 ** 2)
print(f'✔ {SPLIT_ZIP} siap ({size_mb:.1f} MB)')

### 10. Cara Pakai Hasil Split di Notebook Training (V1)

Setelah `NCCT_fixed_split.zip` di-upload ke Google Drive:

1. Di notebook training, **ganti sel download dataset** (sel 3 di V1) dengan download zip split:
   ```python
   !gdown --id <FILE_ID_ZIP_SPLIT>
   !unzip -q NCCT_fixed_split.zip -d NCCT_fixed_split
   ```
2. **Ganti `make_loaders()`** agar membaca dari CSV fixed split (tidak lagi memanggil `patient_level_split()`).
   Fungsi `load_fixed_split()` di sel berikut bisa langsung disalin ke notebook training.

Fungsi pembantu ini memuat CSV split dan mengubah `image_path` relatif menjadi absolut terhadap `split_root`.

In [ ]:
# ==== REFERENSI: cara memuat split FIXED di notebook training (V1) ====
# Salin fungsi ini ke notebook V1 untuk menggantikan split on-the-fly.
def load_fixed_split(split_root='NCCT_fixed_split'):
    """Muat CSV split fixed. image_path (relatif) -> absolut thd split_root."""
    out = {}
    for stage in ['stage1_lvo', 'stage2_laterality', 'stage3_etiology']:
        for sp in ['train', 'val', 'test']:
            fp = os.path.join(split_root, 'metadata', stage, f'{sp}.csv')
            df = pd.read_csv(fp)
            df['image_path'] = df['image_path'].apply(
                lambda p: os.path.join(os.path.abspath(split_root), p))
            out[(stage, sp)] = df
    return out

# Contoh pemakaian (uncomment saat dipakai):
# d = load_fixed_split()
# tr_df, va_df, te_df = d[('stage1_lvo', 'train')], d[('stage1_lvo', 'val')], d[('stage1_lvo', 'test')]